In [ ]:
from pathlib import Path
import astropy
import os, sys
import pandas as pd
import numpy as np
from astropy.table import Table, join,vstack
import desispec.io
import matplotlib.pyplot as plt

# from sklearn.linear_model import LinearRegression
from scipy.stats import binned_statistic
import redrock.templates
from desispec.interpolation import resample_flux
from desispec.resolution import Resolution
from tqdm import tqdm
from matplotlib.gridspec import GridSpec
from utils import better_step
from statsmodels.stats.proportion import proportion_confint

In [ ]:
# figure defaults for AASTEX AJ

COLUMN_WIDTH = 242.26653/72.27 #in inches
TEXT_WIDTH = 513.11743/72.27
SMALL_SIZE = 9 # in pts
NORMAL_SIZE = 10 
BIG_SIZE = 12
FONT_FAMILY = 'Nimbus Roman No9 L'



params = {
    "font.family": FONT_FAMILY,
    "font.size": NORMAL_SIZE,
    "axes.titlesize": NORMAL_SIZE,
    "axes.labelsize": NORMAL_SIZE,
    
    "xtick.labelsize": SMALL_SIZE,
    "ytick.labelsize": SMALL_SIZE,
    "xtick.top": True,
    "ytick.right": True,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "legend.fontsize": NORMAL_SIZE,
    
    "figure.facecolor": "w",
    "figure.dpi": 300,
    'mathtext.fontset' : "cm"
    
}

plt.rcParams.update(params)

In [ ]:
data_path = Path("/global/cfs/cdirs/desi/users/bid13/DESI_II/pilot_obs/MERGED")
cat = Table.read(data_path / "merged_cat_LSST_WL_Y1.fits")
spectra = desispec.io.read_spectra(data_path / "spectra_LSST_WL_Y1.fits")
rr_table = Table.read(data_path / "zbest_LSST_WL_Y1.fits")
field_names = ["XMM","COSMOS", "HERCULES"]

In [ ]:
templates = dict()
for filename in redrock.templates.find_templates():
    t = redrock.templates.Template(filename)
    templates[(t.template_type, t.sub_type)] = t

In [ ]:
template_flux = []
for i, tid in tqdm(enumerate(spectra.fibermap["TARGETID"])):
    rr_row = rr_table[rr_table["TARGETID"]==tid]
    spectype = rr_row["SPECTYPE"].data.astype("U").item()
    subtype = rr_row["SUBTYPE"].data.data.astype("U").item()
    fulltype = (spectype,subtype)
    
    ncoeff = templates[fulltype].flux.shape[0]
    coeff = np.squeeze(rr_row['COEFF'].data)[0:ncoeff]
    tflux = templates[fulltype].flux.T.dot(np.squeeze(coeff.data))
    twave = templates[fulltype].wave * (1+rr_row["Z"])
    R = Resolution(spectra.resolution_data['r'][i])
    txflux = R.dot(resample_flux(spectra.wave['r'], twave, tflux))
    template_flux.append(txflux)
template_flux = np.array(template_flux)

In [ ]:
mask = (spectra.wave["r"]<7500) & (spectra.wave["r"]>5900)
spec = spectra.flux["r"][:,mask]
wave = spectra.wave["r"][mask]
noise = spec-template_flux[:,mask]
signal = np.array([binned_statistic(wave,s,statistic="mean",bins = 64)[0] for s in tqdm(spec)])
noise = np.array([binned_statistic(wave,n,statistic="std",bins = 64)[0] for n in tqdm(noise)])
snr = np.nanmedian(signal/noise, axis =-1)

In [ ]:
snr_table = Table({"TARGETID":spectra.fibermap["TARGETID"].value,"EMP_SNR":snr})
cat = join(cat,snr_table,keys="TARGETID",join_type="left")

In [ ]:
# idx = 0

# plt.plot(spectra.wave["r"],spectra.flux["r"][idx])
# plt.plot(spectra.wave["r"],template_flux[idx])
# plt.axvline(5900,c="k")
# plt.axvline(7500)
# plt.ylim(-2,2)

In [ ]:
def snr_scaling(fiber_mag, exptime):
    return 10**(-1*(fiber_mag-22)/2.5) * (exptime/100)**(0.5)

In [ ]:

k = cat["EMP_SNR"]/snr_scaling(cat["mag_i_fiber"],cat["EXPTIME"])
scale_factor = np.median(k)


In [ ]:
# plt.hist(k,bins=100,histtype="step",range=(-1,2))
# plt.axvline(np.median(k),color="k")
# plt.yscale("log")

# Plots with(out) HERCULES

In [ ]:
with_hercules = True
cat_use = cat.copy()

if  not with_hercules:
    cat = cat_use[cat_use["FIELD_NAME"]!="HERCULES"]

In [ ]:



time_bins=[(70,100),(100,143),(143,204),(204,292),(292,417)]




fig = plt.figure( layout="tight",figsize=(TEXT_WIDTH,0.65*TEXT_WIDTH))
gs = GridSpec(2, 6, figure=fig)
axs = [fig.add_subplot(gs[0, :2]), fig.add_subplot(gs[0, 2:4]), fig.add_subplot(gs[0, 4:6]),
      fig.add_subplot(gs[1, 1:3]), fig.add_subplot(gs[1, 3:5])]


for i, t in enumerate(time_bins):
    cat_plot = cat[(cat["EXPTIME"]/60>=t[0]) & (cat["EXPTIME"]/60<t[1])]
    ratio = cat_plot["EMP_SNR"]/(scale_factor*snr_scaling(cat_plot["mag_i_fiber"],cat_plot["EXPTIME"]))
    axs[i].scatter(cat_plot["mag_i_fiber"], ratio,marker=".",alpha=0.3,s=0.5,c="C0",rasterized=True)
    axs[i].set_ylim(-0.1,2)
    axs[i].set_xlim(21.9,25.1)
    axs[i].set_title(f"{t[0]} $\leq$"+"exp. time [min]"+f"$<${t[1]}")
    _, bins = pd.qcut(cat_plot["mag_i_fiber"],10,retbins=True)
    med, _,_ = binned_statistic(cat_plot["mag_i_fiber"], ratio, bins=bins,statistic="median")
    p25, _, _ = binned_statistic(cat_plot["mag_i_fiber"], ratio, bins=bins,statistic=lambda x: np.percentile(x,25))
    p75,_,_ = binned_statistic(cat_plot["mag_i_fiber"], ratio, bins=bins,statistic=lambda x: np.percentile(x,75))
    better_step(bins,med,(p25,p75),ax=axs[i],c="C1")
    axs[i].axhline(1,ls="--",c="k",lw=0.5)
    axs[i].set_xlabel("$i$-fiber-magnitude")
    
axs[0].set_ylabel(r"$\dfrac{\mathrm{Empirical\ SNR}}{\mathrm{Expected\ SNR}}$")
axs[3].set_ylabel(r"$\dfrac{\mathrm{Empirical\ SNR}}{\mathrm{Expected\ SNR}}$")


if with_hercules:
    plt.savefig("./figs/snr_v_mag_with_hercules.pdf",bbox_inches="tight",dpi=300)
else:
    plt.savefig("./figs/snr_v_mag.pdf",bbox_inches="tight",dpi=300)

In [ ]:
time_bins=[(70,100),(100,143),(143,204),(204,292),(292,417)]

mag_bin_edges = [(22,22.5),(22.5,23),(23,23.5),(23.5,24),(24,24.5),(24.5,25)]


fig,axs = plt.subplots(2,3, layout="tight",figsize=(TEXT_WIDTH,0.65*TEXT_WIDTH))
axs=np.ravel(axs)

for i, t in enumerate(mag_bin_edges):
    cat_plot = cat[(cat["mag_i_fiber"]>=t[0]) & (cat["mag_i_fiber"]<t[1])]
    ratio = cat_plot["EMP_SNR"]/(scale_factor*snr_scaling(cat_plot["mag_i_fiber"],cat_plot["EXPTIME"]))
    axs[i].scatter(cat_plot["EXPTIME"]/60, ratio,marker=".",alpha=0.3,s=0.5,c="C0",rasterized=True)
    axs[i].set_ylim(-0.1,2)
    axs[i].set_xticks([15,100,200,300,400])
    axs[i].set_xlim(-10,410)
    axs[i].set_title(f"{t[0]} $\leq$"+"$i$-fiber-magnitude"+f"$<${t[1]}")
    _, bins = pd.qcut(cat_plot["EXPTIME"]/60,10,retbins=True)
    med, _,_ = binned_statistic(cat_plot["EXPTIME"]/60, ratio, bins=bins,statistic="median")
    p25, _, _ = binned_statistic(cat_plot["EXPTIME"]/60, ratio, bins=bins,statistic=lambda x: np.percentile(x,25))
    p75,_,_ = binned_statistic(cat_plot["EXPTIME"]/60, ratio, bins=bins,statistic=lambda x: np.percentile(x,75))
    better_step(bins,med,(p25,p75),ax=axs[i],c="C1")
    axs[i].axhline(1,ls="--",c="k",lw=0.5)
    axs[i].set_xlabel("Exposure Time [min]")
    
axs[0].set_ylabel(r"$\dfrac{\mathrm{Empirical\ SNR}}{\mathrm{Expected\ SNR}}$")
axs[3].set_ylabel(r"$\dfrac{\mathrm{Empirical\ SNR}}{\mathrm{Expected\ SNR}}$")

if with_hercules:
    plt.savefig("./figs/snr_v_time_with_hercules.pdf",bbox_inches="tight",dpi=300)
else:
    plt.savefig("./figs/snr_v_time.pdf",bbox_inches="tight",dpi=300)

In [ ]:
#plot SNR vs redshift success rate

def make_success_plot(cat,colname="mag_i", nbins=10, ref_sample=None, ax =None, range=None, **kwargs):
    success, bin_edges , _ = binned_statistic(cat[colname], cat["success"],bins=nbins,statistic="mean",range=range)
    sums, bin_edges , _ = binned_statistic(cat[colname], cat["success"],bins=nbins,statistic="sum",range=range)
    count, bin_edges, _ = binned_statistic(cat[colname], cat["success"],bins=nbins,statistic="count",range=range)
    # print(sums)
    # print(count)
    adjustment =1.0
    if ref_sample is not None:
        for i in range(len(bin_edges)-1):
            sel_cat = ref_sample[(ref_sample["HSC_i_MAG"]>=bin_edges[i]) & (ref_sample["HSC_i_MAG"]<bin_edges[i])]
            frac_greater = np.sum(sel_cat["Z"]>1.6)/len(sel_cat)
            print(frac_greater)
    # e_success = np.sqrt(success*(1-success)/(count*adjustment))
    # e_success = 1.96 * e_success # 95% CI
    ci_lo, ci_upp = proportion_confint(sums,count,method="beta")
    
    ax = better_step(bin_edges, success, yerr=(ci_lo,ci_upp), ax = ax,**kwargs)
    # ax.set_ylim(0.3,1)
    # ax.grid(linestyle="--")
    return ax




In [ ]:
cat["success"] = (cat["VI_quality"]>2)
cat = vstack([cat[~cat["success"]],
                      cat[cat["success"] & (cat["Z"]>0.01)]])

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(COLUMN_WIDTH,0.75*COLUMN_WIDTH))
make_success_plot(cat,colname="EMP_SNR",ax=ax,nbins=10,range=(0,4))
ax.set_ylim(0,1.05)
ax.set_xlabel("Empirical SNR")
ax.set_ylabel("Success Rate")

if with_hercules:
    fig.savefig("./figs/snr_v_success_with_hercules.pdf",bbox_inches="tight")
else:
    fig.savefig("./figs/snr_v_success.pdf",bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(COLUMN_WIDTH,0.75*COLUMN_WIDTH))
make_success_plot(cat,colname="DELTACHI2",ax=ax,nbins=20,range=(0,500))
ax.set_ylim(0,1.05)
ax.set_xlabel(r"$\Delta \chi ^2$")
ax.set_ylabel("Success Rate")

if with_hercules:
    fig.savefig("./figs/success_v_deltachi2_with_hercules.pdf",bbox_inches="tight")
else:
    fig.savefig("./figs/success_v_deltachi2.pdf",bbox_inches="tight")

In [ ]:
# fig, ax = plt.subplots(1,1,figsize=(COLUMN_WIDTH,0.5625*COLUMN_WIDTH))
# ax.scatter(cat["DELTACHI2"], cat["EMP_SNR"],s=0.5,alpha=0.1)
# ax.set_ylim(0,4)
# # ax.set_xlim(0,500)
# ax.set_xscale("log")
# # ax.set_yscale("log")